Instalamos librerías para interactuar con una base de datos PostgreSQL

In [ ]:
!pip install pandas
!pip install psycopg2
!pip install sqlalchemy

Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3


ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\e.rangel\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_internal\cli\base_command.py", line 167, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "C:\Users\e.rangel\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_internal\commands\uninstall.py", line 102, in run
    uninstall_pathset.commit()
    ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\e.rangel\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_internal\req\req_uninstall.py", line 420, in commit
    self._moved_paths.commit()
    ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\e.rangel\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_internal\req\req_uninstall.py", line 273, in commit
    save_dir.cleanup()
    ^^^^^^^^^^^^^^^^^^
  File "C:\Users\e.rangel\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_internal\utils\temp_dir.py", line 173, in cleanup
    rm

  Using cached pandas-2.2.2-cp311-cp311-win_amd64.whl (11.6 MB)
     ---------------------------------------- 2.1/2.1 MB 9.5 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.48
    Uninstalling SQLAlchemy-2.0.48:
      Successfully uninstalled SQLAlchemy-2.0.48


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\e.rangel\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\~qlalchemy\\cyextension\\collections.cp311-win_amd64.pyd'
Consider using the `--user` option or check the permissions.

You should consider upgrading via the 'C:\Users\e.rangel\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip' command.


Una vez instaladas, las importamos

In [9]:
import pandas as pd
from sqlalchemy import create_engine

TypeError: <class 'sqlalchemy.engine.result.Result'> is not a generic class

In [3]:
data = {
    'ID':[1, 2, 3],
    'nombre':['danny','carlos','tyrone'],
    'edad':[31,25,8]
}
df = pd.DataFrame(data)
df

,ID,nombre,edad
0,1,danny,31
1,2,carlos,25
2,3,tyrone,8


Establecemos una conexión con nuestra base de datos creada.

In [4]:
host = '34.122.233.40'
port = 5432
database = 'mi_bd_prueba'
user = 'dyanez'
password = '1234'

engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{database}')

Creamos una tabla con la data anteriormente establecida

In [7]:
table_name = 'mi_tabla'
df.to_sql(table_name, engine, index=False, if_exists='replace')

3

Consultamos la tabla creada

In [8]:
consulta_sql = f"SELECT * FROM {table_name}"
df = pd.read_sql(consulta_sql, engine)
df

,ID,nombre,edad
0,1,danny,31
1,2,carlos,25
2,3,tyrone,8


Insertamos más data (desde csv)

In [9]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.9 MB/s eta 0:00:00


In [10]:
from faker import Faker
import random
import csv

rows = []
fake = Faker()
quantity = 2500
for _ in range(quantity):
    id = _ + 1
    nombre = fake.vin()
    cuenta = fake.credit_card_number()
    descripcion = fake.text(max_nb_chars=50)
    categoria = fake.random_element(elements=('Electronics', 'Clothing', 'Home', 'Beauty', 'Toys'))
    stock = random.randint(1,500)
    rows.append([id,nombre,cuenta,descripcion,categoria,stock])

In [11]:
with open('dim_productos.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['id',
                     'nombre',
                     'cuenta',
                     'descripcion',
                     'categoria',
                     'stock'])
    writer.writerows(rows)

In [12]:
df_csv = pd.read_csv('dim_productos.csv')
df_csv

,id,nombre,cuenta,descripcion,categoria,stock
0,1,SF42ZRFE965U4VCHX,6598974045114999,Dog beat bed site why among building.,Clothing,198
1,2,185G3HW19LAHN06GM,4759354579325,Card push art.,Beauty,10
2,3,1NBK0SDD4XCTA2KS0,375744652932070,Participant at civil machine me.,Beauty,120
3,4,NRE2PU9C4EC5VE5Z9,4381687111263293008,Season half analysis. Compare make edge forget.,Clothing,8
4,5,4HC7P9H05YUWHV1BK,3578088570708702,Watch up beyond central.,Clothing,400
...,...,...,...,...,...,...
2495,2496,54S0N8UE4CN08WR7H,3527576535896994,Design rise still else minute beautiful.,Beauty,126
2496,2497,MSHJL2P70GHZT8NG8,3588608897798496,Sell partner ever new choice red style.,Beauty,278
2497,2498,1T56EUD09BR9K6JPW,6011380662139088,Cause member fine ready land company experience.,Home,374
2498,2499,3K68B5HM6PUCYPML9,3549431915607773,Wait happy amount bad.,Clothing,71


In [13]:
df_csv.to_sql('dim_productos', engine, index=False, if_exists='replace')

500

In [16]:
consulta_sql = f"SELECT * FROM dim_productos where stock < 5"
df = pd.read_sql(consulta_sql, engine)
df

,id,nombre,cuenta,descripcion,categoria,stock
0,168,AHTYG1KM2ZLLYGE1J,4329712387470954,Quality life reach question lot let white.,Toys,1
1,287,9LLTH50A9ETWC13NC,30268744409217,Control school share important success analysis.,Electronics,2
2,351,K3FJ4K5S0BEWLKDKL,2675583986163702,Product include town idea put evening recognize.,Home,3
3,440,9KEHHCYSXNUT1ZLND,4000770764589156,Million report its and which budget.,Home,3
4,480,SMZCVBP24T2Z2XCA2,4751908362342005,Possible assume soldier local wife.,Electronics,3
5,498,UZ5LN9WP7VDGKRFWM,5445769952566059,Relationship collection onto.,Clothing,2
6,547,06JBWXBA34MSG55CH,213149734839341,Onto then green appear billion.,Electronics,3
7,571,G69EAYJP06NCJDCNL,3516116150873714,History test long establish eight head but.,Toys,4
8,588,6ZTU49M48D0581D0J,3598353496361697,National land west act individual.,Clothing,4
9,696,FC1Z3JX1XU3J3G5B1,2257796713061157,Crime majority laugh.,Clothing,3
